<a href="https://colab.research.google.com/github/oguzhanguler1/titanic_competition/blob/main/Submission3.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1440]:
import pandas as pd
import numpy as np

from sklearn.model_selection import train_test_split, StratifiedKFold, cross_val_score
from sklearn.metrics import accuracy_score, classification_report, confusion_matrix

from sklearn.svm import SVC
from sklearn.ensemble import RandomForestClassifier, GradientBoostingClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.preprocessing import StandardScaler
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer

In [1441]:
dataset = pd.read_csv("train.csv")

In [1442]:
dataset.isnull().sum().sort_values(ascending=False)

,0
Cabin,687
Age,177
Embarked,2
PassengerId,0
Name,0
Pclass,0
Survived,0
Sex,0
Parch,0
SibSp,0


In [1443]:
dataset.groupby("Sex")["Survived"].agg(["mean", "count"])

,mean,count
Sex,,
female,0.742038,314
male,0.188908,577


In [1444]:
dataset.groupby("Pclass")["Survived"].agg(["mean", "count"])

,mean,count
Pclass,,
1,0.629630,216
2,0.472826,184
3,0.242363,491


In [1445]:
dataset.groupby("SibSp")["Survived"].agg(["mean", "count"])

,mean,count
SibSp,,
0,0.345395,608
1,0.535885,209
2,0.464286,28
3,0.250000,16
4,0.166667,18
5,0.000000,5
8,0.000000,7


In [1446]:
dataset.groupby("Parch")["Survived"].agg(["mean", "count"])

,mean,count
Parch,,
0,0.343658,678
1,0.550847,118
2,0.500000,80
3,0.600000,5
4,0.000000,4
5,0.200000,5
6,0.000000,1


In [1447]:
dataset["HasCabin"] = dataset["Cabin"].notnull().astype(int)

dataset.groupby("HasCabin")["Survived"].agg(["mean", "count"])

,mean,count
HasCabin,,
0,0.299854,687
1,0.666667,204


In [1448]:
dataset["FamilySize"] = dataset["SibSp"] + dataset["Parch"] + 1

In [1449]:
dataset["HasSibSp"] = (dataset["SibSp"] > 0).astype(int)
dataset["HasParch"] = (dataset["Parch"] > 0).astype(int)

In [1450]:
def family_type(row):
    if row["HasSibSp"] == 0 and row["HasParch"] == 0:
        return "Alone"
    elif row["HasSibSp"] == 0 and row["HasParch"] == 1:
        return "ParchOnly"
    elif row["HasSibSp"] == 1 and row["HasParch"] == 0:
        return "SibSpOnly"
    else:
        return "Both"

dataset["FamilyType"] = dataset.apply(family_type, axis=1)

In [1451]:
dataset.groupby("FamilyType")["Survived"].agg(["mean", "count"])

,mean,count
FamilyType,,
Alone,0.303538,537
Both,0.436620,142
ParchOnly,0.661972,71
SibSpOnly,0.496454,141


In [1452]:
dataset["FamilySize"] = dataset["SibSp"] + dataset["Parch"] + 1

def family_group(size):
    if size == 1:
        return "Alone"
    elif size <= 4:
        return "Small"
    else:
        return "Large"

dataset["FamilyGroup"] = dataset["FamilySize"].apply(family_group)

In [1453]:
dataset.groupby(["SibSp", "Parch"])["Survived"].agg(["mean", "count"])

mean  count
SibSp Parch                 
0     0      0.303538    537
      1      0.657895     38
      2      0.724138     29
      3      1.000000      1
      4      0.000000      1
      5      0.000000      2
1     0      0.520325    123
      1      0.596491     57
      2      0.631579     19
      3      0.333333      3
      4      0.000000      3
      5      0.333333      3
      6      0.000000      1
2     0      0.250000     16
      1      0.857143      7
      2      0.500000      4
      3      1.000000      1
3     0      1.000000      2
      1      0.000000      7
      2      0.285714      7
4     1      0.000000      9
      2      0.333333      9
5     2      0.000000      5
8     2      0.000000      7

In [1454]:
dataset["HasSibSp"] = (dataset["SibSp"] > 0).astype(int)
dataset["HasParch"] = (dataset["Parch"] > 0).astype(int)

In [1455]:
dataset.groupby(["HasSibSp", "HasParch"])["Survived"].agg(["mean", "count"])

mean  count
HasSibSp HasParch                 
0        0         0.303538    537
         1         0.661972     71
1        0         0.496454    141
         1         0.436620    142

In [1456]:
from sklearn.impute import SimpleImputer

age_imputer = SimpleImputer(strategy="median")
dataset[["Age"]] = age_imputer.fit_transform(dataset[["Age"]])

dataset["AgeBin"] = pd.cut(
    dataset["Age"],
    bins=[0, 6, 12, 18, 30, 50, 70, np.inf],
    labels=[
        "BabyChild",
        "Child",
        "Teen",
        "YoungAdult",
        "Adult",
        "Older",
        "Senior"
    ],
    include_lowest=True,
    right=False
)

In [1457]:
dataset.groupby("AgeBin")["Survived"].agg(["mean", "count"])

/tmp/ipykernel_3377/3827780487.py:1: FutureWarning: The default of observed=False is deprecated and will be changed to True in a future version of pandas. Pass observed=False to retain current behavior or observed=True to adopt the future default and silence this warning.
  dataset.groupby("AgeBin")["Survived"].agg(["mean", "count"])


,mean,count
AgeBin,,
BabyChild,0.704545,44
Child,0.333333,24
Teen,0.488889,45
YoungAdult,0.328125,448
Adult,0.417969,256
Older,0.388060,67
Senior,0.142857,7


In [1458]:
dataset.groupby(["AgeBin", "Sex"])["Survived"].agg(["mean", "count"])

/tmp/ipykernel_3377/789786224.py:1: FutureWarning: The default of observed=False is deprecated and will be changed to True in a future version of pandas. Pass observed=False to retain current behavior or observed=True to adopt the future default and silence this warning.
  dataset.groupby(["AgeBin", "Sex"])["Survived"].agg(["mean", "count"])


mean  count
AgeBin     Sex                    
BabyChild  female  0.761905     21
           male    0.652174     23
Child      female  0.272727     11
           male    0.384615     13
Teen       female  0.826087     23
           male    0.136364     22
YoungAdult female  0.710345    145
           male    0.145215    303
Adult      female  0.782609     92
           male    0.213415    164
Older      female  0.909091     22
           male    0.133333     45
Senior     female       NaN      0
           male    0.142857      7

In [1459]:
dataset.groupby(["AgeBin", "Pclass"])["Survived"].agg(["mean", "count"])

/tmp/ipykernel_3377/2050605576.py:1: FutureWarning: The default of observed=False is deprecated and will be changed to True in a future version of pandas. Pass observed=False to retain current behavior or observed=True to adopt the future default and silence this warning.
  dataset.groupby(["AgeBin", "Pclass"])["Survived"].agg(["mean", "count"])


mean  count
AgeBin     Pclass                 
BabyChild  1       0.666667      3
           2       1.000000     13
           3       0.571429     28
Child      1       1.000000      1
           2       1.000000      4
           3       0.157895     19
Teen       1       1.000000      8
           2       0.666667      6
           3       0.322581     31
YoungAdult 1       0.602740     73
           2       0.407895     76
           3       0.240803    299
Adult      1       0.701149     87
           2       0.439394     66
           3       0.165049    103
Older      1       0.475000     40
           2       0.333333     18
           3       0.111111      9
Senior     1       0.250000      4
           2       0.000000      1
           3       0.000000      2

In [1460]:
dataset["IsBaby"] = (dataset["Age"] < 6).astype(int)

In [1461]:
dataset.groupby("IsBaby")["Survived"].agg(["mean", "count"])

,mean,count
IsBaby,,
0,0.367178,847
1,0.704545,44


In [1462]:
dataset["SexEncoded"] = dataset["Sex"].map({
    "male": 0,
    "female": 1
})

In [1463]:
dataset["FemaleAfter12"] = (
    (dataset["Age"] >= 12) &
    (dataset["SexEncoded"] == 1)
).astype(int)

In [1464]:
dataset.isnull().sum().sort_values(ascending=False)

,0
Cabin,687
Embarked,2
Pclass,0
Name,0
PassengerId,0
Survived,0
Age,0
Sex,0
SibSp,0
Parch,0


In [1465]:
dataset["IsBaby"] = (dataset["Age"] < 6).astype(int)

In [1466]:
dataset["FemaleAfter12"] = (
    (dataset["Age"] >= 12) &
    (dataset["SexEncoded"] == 1)
).astype(int)

In [1467]:
dataset.groupby("FemaleAfter12")["Survived"].agg(["mean", "count"])

,mean,count
FemaleAfter12,,
0,0.210181,609
1,0.758865,282


In [1468]:
dataset["AgeClassGroup"] = (
    dataset["AgeBin"].astype(str) + "_Pclass" + dataset["Pclass"].astype(str)
)

In [1469]:
dataset.groupby("AgeClassGroup")["Survived"].agg(["mean", "count"]).sort_values(
    by="mean",
    ascending=False
)

,mean,count
AgeClassGroup,,
BabyChild_Pclass2,1.000000,13
Child_Pclass2,1.000000,4
Teen_Pclass1,1.000000,8
Child_Pclass1,1.000000,1
Adult_Pclass1,0.701149,87
BabyChild_Pclass1,0.666667,3
Teen_Pclass2,0.666667,6
YoungAdult_Pclass1,0.602740,73
BabyChild_Pclass3,0.571429,28


In [1470]:
def age_class_risk(group):
    high = [
        "BabyChild_Pclass2",
        "Child_Pclass2",
        "Teen_Pclass1",
        "Child_Pclass1",
        "Adult_Pclass1",
        "BabyChild_Pclass1",
        "Teen_Pclass2",
        "YoungAdult_Pclass1"
    ]

    medium = [
        "BabyChild_Pclass3",
        "Older_Pclass1",
        "Adult_Pclass2",
        "YoungAdult_Pclass2",
        "Older_Pclass2",
        "Teen_Pclass3"
    ]

    low = [
        "Senior_Pclass1",
        "YoungAdult_Pclass3",
        "Adult_Pclass3",
        "Child_Pclass3",
        "Older_Pclass3",
        "Senior_Pclass2",
        "Senior_Pclass3"
    ]

    if group in high:
        return "High"
    elif group in medium:
        return "Medium"
    elif group in low:
        return "Low"
    else:
        return "Unknown"

dataset["AgeClassRisk"] = dataset["AgeClassGroup"].apply(age_class_risk)

In [1471]:
dataset.groupby("AgeClassRisk")["Survived"].agg(["mean", "count"])

,mean,count
AgeClassRisk,,
High,0.702564,195
Low,0.215103,437
Medium,0.428571,259


In [1472]:
dataset.groupby(["AgeClassRisk", "FemaleAfter12"])["Survived"].agg(["mean", "count"])

mean  count
AgeClassRisk FemaleAfter12                 
High         0              0.495575    113
             1              0.987805     82
Low          0              0.125000    328
             1              0.486239    109
Medium       0              0.184524    168
             1              0.879121     91

In [1473]:
dataset.groupby(["AgeClassRisk", "FamilyGroup"])["Survived"].agg(["mean", "count"])

mean  count
AgeClassRisk FamilyGroup                 
High         Alone        0.582418     91
             Large        0.800000      5
             Small        0.808081     99
Low          Alone        0.202572    311
             Large        0.028571     35
             Small        0.329670     91
Medium       Alone        0.348148    135
             Large        0.227273     22
             Small        0.578431    102

In [1474]:
dataset.groupby(["FemaleAfter12", "FamilyGroup"])["Survived"].agg(["mean", "count"])

mean  count
FemaleAfter12 FamilyGroup                 
0             Alone        0.157767    412
              Large        0.051282     39
              Small        0.386076    158
1             Alone        0.784000    125
              Large        0.347826     23
              Small        0.805970    134

In [1475]:
dataset["AgeBinNum"] = pd.cut(
    dataset["Age"],
    bins=[0, 12, 18, 35, 60, 100],
    labels=[0, 1, 2, 3, 4],
    include_lowest=True
).astype(int)

In [1476]:
from sklearn.preprocessing import OrdinalEncoder

numeric_features = [
    "Pclass",
    "SexEncoded",
    "FemaleAfter12",
    "HasCabin",
    "IsBaby"
]

categorical_features = [
    "FamilyGroup",
    "AgeClassRisk"
]

encoder = OrdinalEncoder(
    categories=[
       ["Large", "Alone", "Small"],   # worst -> best
        ["Low", "Medium", "High"]      # AgeClassRisk: worst -> best
    ]
)

encoded_array = encoder.fit_transform(dataset[categorical_features])

encoded_columns = [
    "FamilyGroupEncoded",
    "AgeClassRiskEncoded"
]


encoded_df = pd.DataFrame(
    encoded_array,
    columns=encoded_columns,
    index=dataset.index
)

X_numeric = dataset[numeric_features]

X = pd.concat(
    [X_numeric, encoded_df],
    axis=1
)

y = dataset["Survived"]

In [1477]:
from sklearn.model_selection import train_test_split
from sklearn.svm import SVC
from sklearn.metrics import accuracy_score, classification_report, confusion_matrix

X_train, X_val, y_train, y_val = train_test_split(
    X,
    y,
    test_size=0.1,
    random_state=42,
    stratify=y
)

model = SVC()

model.fit(X_train, y_train)

pred = model.predict(X_val)

print("Accuracy:", accuracy_score(y_val, pred))
print(classification_report(y_val, pred))
print(confusion_matrix(y_val, pred))

Accuracy: 0.7888888888888889
              precision    recall  f1-score   support

           0       0.80      0.87      0.83        55
           1       0.77      0.66      0.71        35

    accuracy                           0.79        90
   macro avg       0.78      0.76      0.77        90
weighted avg       0.79      0.79      0.79        90

[[48  7]
 [12 23]]


In [1478]:
from sklearn.model_selection import StratifiedKFold, cross_val_score

skf = StratifiedKFold(
    n_splits=5,
    shuffle=True,
    random_state=42
)

scores = cross_val_score(
    SVC(),
    X,
    y,
    cv=skf,
    scoring="accuracy"
)

print("CV Scores:", scores)
print("Mean Accuracy:", scores.mean())
print("Std:", scores.std())

CV Scores: [0.83240223 0.8258427  0.81460674 0.81460674 0.80898876]
Mean Accuracy: 0.8192894356914191
Std: 0.008542245722102606


In [1479]:
def feature_effect(df, feature):
    result = df.groupby(feature)["Survived"].agg(["mean", "count"])
    result = result.sort_values(by="mean", ascending=False)
    return result

In [1480]:
candidate_features = [
    "Sex",
    "SexEncoded",
    "Pclass",
    "AgeBin",
    "AgeBinNum",
    "IsBaby",
    "FemaleAfter12",
    "SibSp",
    "Parch",
    "FamilySize",
    "FamilyGroup",
    "HasCabin",
    "AgeClassRisk"
]

In [1481]:
for feature in candidate_features:
    print("\n" + "="*50)
    print(feature)
    print("="*50)
    print(feature_effect(dataset, feature))


Sex
            mean  count
Sex                    
female  0.742038    314
male    0.188908    577

SexEncoded
                mean  count
SexEncoded                 
1           0.742038    314
0           0.188908    577

Pclass
            mean  count
Pclass                 
1       0.629630    216
2       0.472826    184
3       0.242363    491

AgeBin
                mean  count
AgeBin                     
BabyChild   0.704545     44
Teen        0.488889     45
Adult       0.417969    256
Older       0.388060     67
Child       0.333333     24
YoungAdult  0.328125    448
Senior      0.142857      7

AgeBinNum
               mean  count
AgeBinNum                 
0          0.579710     69
1          0.428571     70
3          0.400000    195
2          0.353271    535
4          0.227273     22

IsBaby
            mean  count
IsBaby                 
1       0.704545     44
0       0.367178    847

FemaleAfter12
                   mean  count
FemaleAfter12                 
1     

/tmp/ipykernel_3377/3458623621.py:2: FutureWarning: The default of observed=False is deprecated and will be changed to True in a future version of pandas. Pass observed=False to retain current behavior or observed=True to adopt the future default and silence this warning.
  result = df.groupby(feature)["Survived"].agg(["mean", "count"])


In [1482]:
from sklearn.model_selection import StratifiedKFold, cross_val_score
from sklearn.svm import SVC
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier, GradientBoostingClassifier, HistGradientBoostingClassifier
from sklearn.neighbors import KNeighborsClassifier
from sklearn.neural_network import MLPClassifier
from sklearn.preprocessing import StandardScaler
from sklearn.pipeline import Pipeline
import pandas as pd

skf = StratifiedKFold(
    n_splits=5,
    shuffle=True,
    random_state=42
)

models = {
    "SVC": SVC(),

    "Scaled SVC": Pipeline([
        ("scaler", StandardScaler()),
        ("model", SVC())
    ]),

    "Logistic Regression": Pipeline([
        ("scaler", StandardScaler()),
        ("model", LogisticRegression(max_iter=1000))
    ]),

    "Random Forest": RandomForestClassifier(
        n_estimators=200,
        random_state=42
    ),

    "Gradient Boosting": GradientBoostingClassifier(
        random_state=42
    ),

    "Hist Gradient Boosting": HistGradientBoostingClassifier(
        random_state=42
    ),

    "KNN": Pipeline([
        ("scaler", StandardScaler()),
        ("model", KNeighborsClassifier(n_neighbors=5))
    ]),

    "MLP": Pipeline([
        ("scaler", StandardScaler()),
        ("model", MLPClassifier(
            hidden_layer_sizes=(32, 16),
            max_iter=1000,
            random_state=42
        ))
    ])
}

results = []

for name, model in models.items():
    scores = cross_val_score(
        model,
        X,
        y,
        cv=skf,
        scoring="accuracy"
    )

    results.append({
        "Model": name,
        "Mean Accuracy": scores.mean(),
        "Std": scores.std(),
        "Scores": scores
    })

results_df = pd.DataFrame(results).sort_values(
    by="Mean Accuracy",
    ascending=False
)

results_df

,Model,Mean Accuracy,Std,Scores
2,Logistic Regression,0.827155,0.011538,"[0.8324022346368715, 0.8314606741573034, 0.820..."
6,KNN,0.822648,0.015147,"[0.8435754189944135, 0.8146067415730337, 0.803..."
1,Scaled SVC,0.821537,0.011073,"[0.8324022346368715, 0.8146067415730337, 0.808..."
7,MLP,0.820419,0.010165,"[0.8268156424581006, 0.8146067415730337, 0.808..."
0,SVC,0.819289,0.008542,"[0.8324022346368715, 0.8258426966292135, 0.814..."
5,Hist Gradient Boosting,0.818172,0.012234,"[0.8268156424581006, 0.8089887640449438, 0.803..."
3,Random Forest,0.814808,0.011343,"[0.8212290502793296, 0.8089887640449438, 0.797..."
4,Gradient Boosting,0.813671,0.014627,"[0.8324022346368715, 0.7921348314606742, 0.803..."


In [1483]:
from sklearn.neighbors import KNeighborsClassifier
from sklearn.preprocessing import StandardScaler
from sklearn.pipeline import Pipeline
from sklearn.model_selection import StratifiedKFold, cross_val_score
import pandas as pd

skf = StratifiedKFold(
    n_splits=5,
    shuffle=True,
    random_state=42
)

knn_results = []

for k in [3, 5, 7, 9, 11, 13, 15, 17, 19]:
    knn_model = Pipeline([
        ("scaler", StandardScaler()),
        ("model", KNeighborsClassifier(n_neighbors=k))
    ])

    scores = cross_val_score(
        knn_model,
        X,
        y,
        cv=skf,
        scoring="accuracy"
    )

    knn_results.append({
        "k": k,
        "Mean Accuracy": scores.mean(),
        "Std": scores.std(),
        "Scores": scores
    })

knn_results_df = pd.DataFrame(knn_results).sort_values(
    by="Mean Accuracy",
    ascending=False
)

knn_results_df

,k,Mean Accuracy,Std,Scores
1,5,0.822648,0.015147,"[0.8435754189944135, 0.8146067415730337, 0.803..."
0,3,0.821518,0.020759,"[0.8491620111731844, 0.8033707865168539, 0.797..."
2,7,0.821512,0.021975,"[0.8547486033519553, 0.8089887640449438, 0.792..."
5,13,0.820419,0.013382,"[0.8268156424581006, 0.8146067415730337, 0.803..."
3,9,0.819296,0.013113,"[0.8268156424581006, 0.8202247191011236, 0.797..."
7,17,0.818178,0.016160,"[0.8212290502793296, 0.8258426966292135, 0.803..."
4,11,0.817049,0.013710,"[0.8268156424581006, 0.8202247191011236, 0.792..."
6,15,0.815931,0.015713,"[0.8212290502793296, 0.8146067415730337, 0.803..."
8,19,0.812573,0.014864,"[0.8100558659217877, 0.8202247191011236, 0.797..."


In [1484]:
import pandas as pd
import numpy as np

from sklearn.linear_model import LogisticRegression
from sklearn.neighbors import KNeighborsClassifier
from sklearn.preprocessing import StandardScaler, OrdinalEncoder
from sklearn.pipeline import Pipeline

# Read test data
test_dataset = pd.read_csv("test.csv")

test_passenger_ids = test_dataset["PassengerId"]

# -----------------------------
# Train-side final feature setup
# -----------------------------

numeric_features = [
    "Pclass",
    "SexEncoded",
    "FemaleAfter12",
    "HasCabin",
    "IsBaby"
]

categorical_features = [
    "FamilyGroup",
    "AgeClassRisk"
]

# Refit encoder on full train data
encoder = OrdinalEncoder(
    categories=[
        ["Large", "Alone", "Small"],   # worst -> best
        ["Low", "Medium", "High"]      # worst -> best
    ]
)

encoded_array = encoder.fit_transform(dataset[categorical_features])

encoded_df = pd.DataFrame(
    encoded_array,
    columns=["FamilyGroupEncoded", "AgeClassRiskEncoded"],
    index=dataset.index
)

X_numeric = dataset[numeric_features]

X_final = pd.concat(
    [X_numeric, encoded_df],
    axis=1
)

y_final = dataset["Survived"]

# -----------------------------
# Test preprocessing
# -----------------------------

# Age imputation using TRAIN median
age_median = dataset["Age"].median()
test_dataset["Age"] = test_dataset["Age"].fillna(age_median)

# Sex encoding
test_dataset["SexEncoded"] = test_dataset["Sex"].map({
    "male": 0,
    "female": 1
})

# IsBaby
test_dataset["IsBaby"] = (test_dataset["Age"] < 6).astype(int)

# FemaleAfter12
test_dataset["FemaleAfter12"] = (
    (test_dataset["Age"] >= 12) &
    (test_dataset["SexEncoded"] == 1)
).astype(int)

# HasCabin
test_dataset["HasCabin"] = test_dataset["Cabin"].notnull().astype(int)

# AgeBin for AgeClassRisk
test_dataset["AgeBin"] = pd.cut(
    test_dataset["Age"],
    bins=[0, 6, 12, 18, 30, 50, 70, np.inf],
    labels=[
        "BabyChild",
        "Child",
        "Teen",
        "YoungAdult",
        "Adult",
        "Older",
        "Senior"
    ],
    include_lowest=True,
    right=False
)

# AgeClassGroup
test_dataset["AgeClassGroup"] = (
    test_dataset["AgeBin"].astype(str) + "_Pclass" + test_dataset["Pclass"].astype(str)
)

# AgeClassRisk
def age_class_risk(group):
    high = [
        "BabyChild_Pclass2",
        "Child_Pclass2",
        "Teen_Pclass1",
        "Child_Pclass1",
        "Adult_Pclass1",
        "BabyChild_Pclass1",
        "Teen_Pclass2",
        "YoungAdult_Pclass1"
    ]

    medium = [
        "BabyChild_Pclass3",
        "Older_Pclass1",
        "Adult_Pclass2",
        "YoungAdult_Pclass2",
        "Older_Pclass2",
        "Teen_Pclass3"
    ]

    low = [
        "Senior_Pclass1",
        "YoungAdult_Pclass3",
        "Adult_Pclass3",
        "Child_Pclass3",
        "Older_Pclass3",
        "Senior_Pclass2",
        "Senior_Pclass3"
    ]

    if group in high:
        return "High"
    elif group in medium:
        return "Medium"
    elif group in low:
        return "Low"
    else:
        return "Medium"   # safer fallback

test_dataset["AgeClassRisk"] = test_dataset["AgeClassGroup"].apply(age_class_risk)

# FamilySize
test_dataset["FamilySize"] = test_dataset["SibSp"] + test_dataset["Parch"] + 1

# FamilyGroup
def family_group(size):
    if size == 1:
        return "Alone"
    elif size <= 4:
        return "Small"
    else:
        return "Large"

test_dataset["FamilyGroup"] = test_dataset["FamilySize"].apply(family_group)

# Encode categorical test features using TRAIN encoder
test_encoded_array = encoder.transform(test_dataset[categorical_features])

test_encoded_df = pd.DataFrame(
    test_encoded_array,
    columns=["FamilyGroupEncoded", "AgeClassRiskEncoded"],
    index=test_dataset.index
)

X_test_numeric = test_dataset[numeric_features]

X_test_final = pd.concat(
    [X_test_numeric, test_encoded_df],
    axis=1
)

# Make sure column order is same
X_test_final = X_test_final[X_final.columns]

# -----------------------------
# Logistic Regression submission
# -----------------------------

logistic_model = Pipeline([
    ("scaler", StandardScaler()),
    ("model", LogisticRegression(max_iter=1000))
])

logistic_model.fit(X_final, y_final)

logistic_predictions = logistic_model.predict(X_test_final)

submission_logistic = pd.DataFrame({
    "PassengerId": test_passenger_ids,
    "Survived": logistic_predictions
})

submission_logistic.to_csv("submission3_logistic.csv", index=False)

submission_logistic.head()

,PassengerId,Survived
0,892,0
1,893,1
2,894,0
3,895,0
4,896,1


In [1485]:
# -----------------------------
# KNN submission
# -----------------------------

knn_model = Pipeline([
    ("scaler", StandardScaler()),
    ("model", KNeighborsClassifier(n_neighbors=5))
])

knn_model.fit(X_final, y_final)

knn_predictions = knn_model.predict(X_test_final)

submission_knn = pd.DataFrame({
    "PassengerId": test_passenger_ids,
    "Survived": knn_predictions
})

submission_knn.to_csv("submission3_knn.csv", index=False)

submission_knn.head()

,PassengerId,Survived
0,892,0
1,893,0
2,894,0
3,895,0
4,896,0
